In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lhuythc/output-bartpho/__huggingface_repos__.json
/kaggle/input/datasets/lhuythc/output-bartpho/outputs/eval_finetune_bartpho_word.csv
/kaggle/input/datasets/lhuythc/output-bartpho/outputs/experiment_config_bartpho.json
/kaggle/input/datasets/lhuythc/output-bartpho/outputs/eval_zeroshot_bartpho_word.csv
/kaggle/input/datasets/lhuythc/output-bartpho/outputs/improvement_bartpho_correct.csv
/kaggle/input/datasets/lhuythc/output-bartpho/outputs/finetune_bartpho_word_correct.csv
/kaggle/input/datasets/lhuythc/output-bartpho/outputs/eval_finetune_bartpho_syllable.csv
/kaggle/input/datasets/lhuythc/output-bartpho/outputs/eval_zeroshot_bartpho_syllable.csv
/kaggle/input/datasets/lhuythc/output-bartpho/outputs/zeroshot_bartpho_word_correct.csv
/kaggle/input/datasets/lhuythc/output-bartpho/outputs/finetune_bartpho_syllable_correct.csv
/kaggle/input/datasets/lhuythc/output-bartpho/outputs/zeroshot_bartpho_syllable_correct.csv
/kaggle/input/datasets/lhuythc/output-bartpho/ou

In [2]:
# ============================================================
# OFFICIAL TEST INFERENCE - BARTpho-syllable LoRA - GPU VERSION
# Kaggle: bật Accelerator = GPU
# ============================================================

!pip install -q transformers peft sentencepiece evaluate rouge_score accelerate pandas pyarrow

# ============================================================
# 0. ENV - PHẢI ĐẶT TRƯỚC KHI IMPORT TORCH
# ============================================================

import os

# Dùng GPU số 0. Nếu Kaggle T4x2 thì vẫn dùng 1 GPU là đủ ổn cho inference.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import gc
import time
import torch
import pandas as pd
import evaluate

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel


# ============================================================
# 1. CONFIG
# ============================================================

TEST_PATH = "/kaggle/input/datasets/lhuythc/test-official/test-00000-of-00001 (1).parquet"

BASE_MODEL_NAME = "vinai/bartpho-syllable"

ADAPTER_DIR = "/kaggle/input/datasets/lhuythc/output-bartpho/checkpoints/bartpho_syllable_lora_correct"

# Nếu path trên không tồn tại, thử path này:
# ADAPTER_DIR = "/kaggle/input/datasets/lhuythc/output-bartpho/bartpho_correct_results/kaggle/working/checkpoints/bartpho_syllable_lora_correct"

OUTPUT_DIR = "/kaggle/working/official_test_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PRED_PATH = os.path.join(
    OUTPUT_DIR,
    "official_test_bartpho_syllable_lora_predictions.csv"
)

OUTPUT_EVAL_PATH = os.path.join(
    OUTPUT_DIR,
    "official_test_bartpho_syllable_lora_eval.csv"
)

OUTPUT_METRICS_PATH = os.path.join(
    OUTPUT_DIR,
    "official_test_bartpho_syllable_lora_metrics.csv"
)

MAX_INPUT_LENGTH = 1024
MAX_OUTPUT_LENGTH = 200
MIN_OUTPUT_LENGTH = 20
NUM_BEAMS = 2

# Batch size cho GPU.
# Nếu bị CUDA out of memory thì giảm xuống 2.
# Nếu chạy ổn, có thể tăng lên 8.
BATCH_SIZE = 4


# ============================================================
# 2. DEVICE CHECK
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "Notebook hiện chưa nhận GPU. Hãy vào Kaggle Settings -> Accelerator -> chọn GPU, "
        "sau đó Restart Session và chạy lại."
    )

DEVICE = torch.device("cuda")
DTYPE = torch.float16

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)


# ============================================================
# 3. READ TEST FILE
# ============================================================

def read_any(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Không tìm thấy file test: {path}")

    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    elif path.endswith(".csv"):
        return pd.read_csv(path)
    elif path.endswith(".json"):
        return pd.read_json(path)
    elif path.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    else:
        raise ValueError(f"Unsupported file type: {path}")


def clean_test_df(df, text_col="article", summary_col="summary"):
    df = df.copy()

    if text_col not in df.columns:
        raise ValueError(
            f"Không tìm thấy cột '{text_col}'. Columns hiện có: {df.columns.tolist()}"
        )

    df[text_col] = df[text_col].astype(str).str.strip()
    df = df[df[text_col].str.len() > 50].reset_index(drop=True)

    if summary_col in df.columns:
        df[summary_col] = df[summary_col].astype(str).str.strip()
        df = df.rename(columns={text_col: "article", summary_col: "summary"})
        return df[["article", "summary"]].reset_index(drop=True)
    else:
        df = df.rename(columns={text_col: "article"})
        return df[["article"]].reset_index(drop=True)


raw_test_df = read_any(TEST_PATH)
official_test_df = clean_test_df(raw_test_df)

print("Official test shape:", official_test_df.shape)
print("Columns:", official_test_df.columns.tolist())
display(official_test_df.head())


# ============================================================
# 4. HELPER: FIND ADAPTER IF PATH WRONG
# ============================================================

def check_adapter_dir(adapter_dir):
    if not os.path.exists(adapter_dir):
        print("ADAPTER_DIR hiện tại không tồn tại:", adapter_dir)
        print("Đang tìm adapter_model.safetensors trong /kaggle/input ...")

        found = []
        for root, dirs, files in os.walk("/kaggle/input"):
            if "adapter_model.safetensors" in files:
                found.append(root)

        if len(found) == 0:
            raise FileNotFoundError(
                "Không tìm thấy adapter_model.safetensors trong /kaggle/input. "
                "Bạn cần Add Dataset chứa output LoRA vào notebook."
            )

        print("Tìm thấy adapter ở các path sau:")
        for p in found:
            print(p)

        raise FileNotFoundError(
            "Hãy copy một trong các path trên và gán lại vào biến ADAPTER_DIR."
        )

    adapter_file = os.path.join(adapter_dir, "adapter_model.safetensors")
    if not os.path.exists(adapter_file):
        raise FileNotFoundError(
            f"Không tìm thấy adapter_model.safetensors trong: {adapter_dir}"
        )

    print("Adapter OK:", adapter_dir)


check_adapter_dir(ADAPTER_DIR)


# ============================================================
# 5. GENERATE WITH BARTpho-syllable LoRA ON GPU
# ============================================================

def generate_bartpho_syllable_lora_gpu(
    test_df,
    base_model_name,
    adapter_dir,
    output_path,
    batch_size=4
):
    print("=" * 80)
    print("BARTpho-syllable LoRA OFFICIAL TEST INFERENCE - GPU")
    print("Base model:", base_model_name)
    print("Adapter:", adapter_dir)
    print("Test size:", len(test_df))
    print("Batch size:", batch_size)
    print("Device:", DEVICE)
    print("=" * 80)

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_name,
        use_fast=False
    )

    print("Loading base model in FP16...")
    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        base_model_name,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True
    )

    print("Loading LoRA adapter...")
    model = PeftModel.from_pretrained(
        base_model,
        adapter_dir
    )

    print("Merging LoRA adapter...")
    model = model.merge_and_unload()

    model.to(DEVICE)
    model.eval()

    results = []
    total = len(test_df)

    start_all = time.time()

    for start_idx in range(0, total, batch_size):
        end_idx = min(start_idx + batch_size, total)
        batch_df = test_df.iloc[start_idx:end_idx]

        articles = batch_df["article"].astype(str).tolist()

        # BARTpho-syllable dùng raw Vietnamese text, KHÔNG word_tokenize
        inputs = tokenizer(
            articles,
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )

        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        torch.cuda.synchronize()
        start_batch = time.time()

        with torch.no_grad():
            with torch.cuda.amp.autocast(dtype=torch.float16):
                output_ids = model.generate(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_length=MAX_OUTPUT_LENGTH,
                    min_length=MIN_OUTPUT_LENGTH,
                    num_beams=NUM_BEAMS,
                    no_repeat_ngram_size=3,
                    early_stopping=True,
                    length_penalty=1.0
                )

        torch.cuda.synchronize()
        elapsed_batch = time.time() - start_batch

        predictions = tokenizer.batch_decode(
            output_ids,
            skip_special_tokens=True
        )

        time_per_sample = elapsed_batch / len(batch_df)

        for local_i, (_, row) in enumerate(batch_df.iterrows()):
            item = {
                "id": int(start_idx + local_i),
                "article": str(row["article"]),
                "prediction": predictions[local_i].strip(),
                "inference_time_sec": time_per_sample,
                "base_model": base_model_name,
                "adapter": adapter_dir,
                "mode": "syllable_lora_gpu"
            }

            if "summary" in test_df.columns:
                item["reference"] = str(row["summary"])

            results.append(item)

        done = end_idx
        avg_time = sum(x["inference_time_sec"] for x in results) / len(results)
        print(f"Done {done}/{total} | avg = {avg_time:.2f}s/sample")

        # Dọn nhẹ cache sau mỗi vài batch
        if done % (batch_size * 10) == 0:
            torch.cuda.empty_cache()

    out_df = pd.DataFrame(results)
    out_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    total_time = time.time() - start_all

    print("=" * 80)
    print("Saved prediction file:", output_path)
    print("Total time:", total_time)
    print("Average time:", out_df["inference_time_sec"].mean())
    print("=" * 80)

    del model
    del base_model
    gc.collect()
    torch.cuda.empty_cache()

    return out_df


pred_df = generate_bartpho_syllable_lora_gpu(
    test_df=official_test_df,
    base_model_name=BASE_MODEL_NAME,
    adapter_dir=ADAPTER_DIR,
    output_path=OUTPUT_PRED_PATH,
    batch_size=BATCH_SIZE
)

display(pred_df.head())


# ============================================================
# 6. COMPUTE ROUGE IF TEST HAS SUMMARY
# ============================================================

rouge = evaluate.load("rouge")

if "reference" in pred_df.columns:
    predictions = pred_df["prediction"].fillna("").astype(str).tolist()
    references = pred_df["reference"].fillna("").astype(str).tolist()

    scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=False
    )

    metrics = {
        "model": "bartpho_syllable_lora_official_test_gpu",
        "num_test_samples": len(pred_df),
        "rouge1": float(scores["rouge1"]),
        "rouge2": float(scores["rouge2"]),
        "rougeL": float(scores["rougeL"]),
        "rougeLsum": float(scores["rougeLsum"]),
        "avg_time_sec_per_sample": float(pred_df["inference_time_sec"].mean()),
        "batch_size": BATCH_SIZE,
        "device": str(DEVICE),
        "gpu_name": torch.cuda.get_device_name(0)
    }

    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(
        OUTPUT_METRICS_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    eval_df = pd.DataFrame()
    eval_df["id"] = pred_df["id"]
    eval_df["prediction"] = pred_df["prediction"]
    eval_df["reference"] = pred_df["reference"]
    eval_df["source"] = pred_df["article"]

    eval_df.to_csv(
        OUTPUT_EVAL_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print("=" * 80)
    print("OFFICIAL TEST METRICS")
    print(metrics)
    print("Saved eval file:", OUTPUT_EVAL_PATH)
    print("Saved metrics file:", OUTPUT_METRICS_PATH)

    display(metrics_df)

else:
    submit_df = pd.DataFrame()
    submit_df["id"] = pred_df["id"]
    submit_df["prediction"] = pred_df["prediction"]

    submit_path = os.path.join(
        OUTPUT_DIR,
        "official_test_bartpho_syllable_lora_submit.csv"
    )

    submit_df.to_csv(
        submit_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Official test không có summary/reference.")
    print("Chỉ xuất file prediction để nộp:", submit_path)
    display(submit_df.head())


# ============================================================
# 7. SHOW OUTPUT FILES
# ============================================================

print("Output files:")
for f in os.listdir(OUTPUT_DIR):
    print(os.path.join(OUTPUT_DIR, f))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

,article,summary
0,Văn phòng mới của MayTrip tại địa chỉ 833 Lê H...,MayTrip khai trương văn phòng mới tại TP. HCM ...
1,"Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...",Rừng hoa dã quỳ ở Vườn quốc gia Ba Vì rộng kho...
2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...","Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ..."
3,"Cầm cốc chocolate đen nóng trên tay, nữ du khá...",Du khách Anh Monisha Rajesh bắt đầu hành trình...
4,"Cồn Én nằm giữa sông Tiền, thuộc xã Tấn Mỹ, rộ...",Cồn Én là một điểm đến du lịch nằm giữa sông T...


Adapter OK: /kaggle/input/datasets/lhuythc/output-bartpho/checkpoints/bartpho_syllable_lora_correct
BARTpho-syllable LoRA OFFICIAL TEST INFERENCE - GPU
Base model: vinai/bartpho-syllable
Adapter: /kaggle/input/datasets/lhuythc/output-bartpho/checkpoints/bartpho_syllable_lora_correct
Test size: 1344
Batch size: 4
Device: cuda
Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model in FP16...


...

Done 1344/1344 | avg = 1.03s/sample
Saved prediction file: /kaggle/working/official_test_outputs/official_test_bartpho_syllable_lora_predictions.csv
Total time: 1390.3241348266602
Average time: 1.0307304912379809


,id,article,prediction,inference_time_sec,base_model,adapter,mode,reference
0,0,Văn phòng mới của MayTrip tại địa chỉ 833 Lê H...,Công ty lữ hành MayTrip vừa khai trương văn ph...,1.895772,vinai/bartpho-syllable,/kaggle/input/datasets/lhuythc/output-bartpho/...,syllable_lora_gpu,MayTrip khai trương văn phòng mới tại TP. HCM ...
1,1,"Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...","Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...",1.895772,vinai/bartpho-syllable,/kaggle/input/datasets/lhuythc/output-bartpho/...,syllable_lora_gpu,Rừng hoa dã quỳ ở Vườn quốc gia Ba Vì rộng kho...
2,2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...","Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...",1.895772,vinai/bartpho-syllable,/kaggle/input/datasets/lhuythc/output-bartpho/...,syllable_lora_gpu,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ..."
3,3,"Cầm cốc chocolate đen nóng trên tay, nữ du khá...",Nữ du khách Anh Monisha Rajesh bước ra khỏi qu...,1.895772,vinai/bartpho-syllable,/kaggle/input/datasets/lhuythc/output-bartpho/...,syllable_lora_gpu,Du khách Anh Monisha Rajesh bắt đầu hành trình...
4,4,"Cồn Én nằm giữa sông Tiền, thuộc xã Tấn Mỹ, rộ...",Khu du lịch sinh thái Cồn Én nằm giữa sông Tiề...,1.050832,vinai/bartpho-syllable,/kaggle/input/datasets/lhuythc/output-bartpho/...,syllable_lora_gpu,Cồn Én là một điểm đến du lịch nằm giữa sông T...


OFFICIAL TEST METRICS
{'model': 'bartpho_syllable_lora_official_test_gpu', 'num_test_samples': 1344, 'rouge1': 0.699729011788773, 'rouge2': 0.43523366396648766, 'rougeL': 0.44998872110423027, 'rougeLsum': 0.4523648955094067, 'avg_time_sec_per_sample': 1.0307304912379809, 'batch_size': 4, 'device': 'cuda', 'gpu_name': 'Tesla T4'}
Saved eval file: /kaggle/working/official_test_outputs/official_test_bartpho_syllable_lora_eval.csv
Saved metrics file: /kaggle/working/official_test_outputs/official_test_bartpho_syllable_lora_metrics.csv


,model,num_test_samples,rouge1,rouge2,rougeL,rougeLsum,avg_time_sec_per_sample,batch_size,device,gpu_name
0,bartpho_syllable_lora_official_test_gpu,1344,0.699729,0.435234,0.449989,0.452365,1.03073,4,cuda,Tesla T4


Output files:
/kaggle/working/official_test_outputs/official_test_bartpho_syllable_lora_metrics.csv
/kaggle/working/official_test_outputs/official_test_bartpho_syllable_lora_predictions.csv
/kaggle/working/official_test_outputs/official_test_bartpho_syllable_lora_eval.csv
